In [ ]:
import os
import pandas as pd
from fredapi import Fred
from dotenv import load_dotenv

# =============================================================================
# CONFIG
# =============================================================================

load_dotenv()
FRED_KEY = os.getenv("FRED_KEY")

START = "2019-01-01"
END   = "2026-01-01"

OUTPUT_PATH = "../data/raw/macro.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

fred = Fred(api_key=FRED_KEY)

# =============================================================================
# FRED SERIES
# =============================================================================

fred_series = {
    "FED_RATE_USD": "FEDFUNDS",
    "US10Y": "DGS10",
    "US2Y": "DGS2",
    "US3M": "DGS3MO",
    "TED_SPREAD": "TEDRATE",
    "BBB_SPREAD": "BAA10YM",
    "T10Y_IE": "T10YIE",
}

# =============================================================================
# FETCH
# =============================================================================

data = {}

for name, code in fred_series.items():
    s = fred.get_series(code, observation_start=START, observation_end=END)
    s.index = pd.to_datetime(s.index)
    data[name] = s
    print(f"Loaded {name:15s} ({len(s):,} rows)")

macro = pd.DataFrame(data).sort_index()

# =============================================================================
# DERIVED FEATURES
# =============================================================================

macro["YIELD_CURVE_SLOPE"] = macro["US10Y"] - macro["US2Y"]

# =============================================================================
# CLEAN + SAVE
# =============================================================================

macro = macro.ffill()

macro.to_csv(OUTPUT_PATH)
print(f"\nSaved macro data → {OUTPUT_PATH}")
print(f"Final shape: {macro.shape}")

In [ ]:
import pandas as pd
import os

# =============================================================================
# PATHS
# =============================================================================

INPUT_PATH  = "../data/raw/macro.csv"
OUTPUT_PATH = "../data/processed/macro_D.csv"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# =============================================================================
# LOAD (unnamed index = date)
# =============================================================================

df = pd.read_csv(INPUT_PATH, index_col=0)
df.index = pd.to_datetime(df.index)

# ADD DATE COLUMN
df = df.reset_index().rename(columns={"index": "date"})
df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved cleaned macro file → {OUTPUT_PATH}")
print(df.head())

In [ ]:
import pandas as pd
import numpy as np

# ============================================================================
# CONFIG
# ============================================================================

DATASETS = {
    "DXY": "../data/raw/DXY.csv",
    "ES": "../data/raw/ES.csv",
}

# ============================================================================
# RUN ON ALL FUTURES
# ============================================================================

for name, path in DATASETS.items():

    print(f"\n{'='*80}")
    print(f"PROCESSING {name}")
    print(f"{'='*80}")

    df = pd.read_csv(path)
    print(f"Raw data: {len(df):,} rows")
    print(f"Columns: {list(df.columns)}")

    print("\nSample data:")
    print(df.head())

    # ------------------------------------------------------------------------
    # SYMBOL ANALYSIS
    # ------------------------------------------------------------------------

    unique_symbols = df["symbol"].unique()
    print(f"\nTotal unique symbols: {len(unique_symbols)}")

    print("\nTop symbols by frequency:")
    print(df["symbol"].value_counts().head(20))

    # ------------------------------------------------------------------------
    # REMOVE CALENDAR SPREADS
    # ------------------------------------------------------------------------

    df_no_spreads = df[~df["symbol"].str.contains("-", regex=False)].copy()

    print(f"\nActual contracts (no spreads): {df_no_spreads['symbol'].nunique()}")
    print("Contracts:")
    print(sorted(df_no_spreads["symbol"].unique()))

    # ------------------------------------------------------------------------
    # VOLUME STATS
    # ------------------------------------------------------------------------

    print("\nVolume range:")
    print(f"  Min: {df['volume'].min():,.0f}")
    print(f"  Max: {df['volume'].max():,.0f}")
    print(f"  Mean: {df['volume'].mean():,.0f}")
    print(f"  Median: {df['volume'].median():,.0f}")

In [ ]:
import pandas as pd
import os

# =============================================================================
# CONFIG
# =============================================================================

DATASETS = {
    "ES": "../data/raw/ES.csv",
    "DXY": "../data/raw/DXY.csv",
}

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = "2019-01-01"
END_DATE   = "2026-01-01"

# =============================================================================
# HELPERS
# =============================================================================

def assign_futures_trading_day(ts):
    """
    Futures trading day rolls at 23:00 UTC.
    """
    return (ts - pd.Timedelta(hours=23)).dt.date


# =============================================================================
# MAIN LOOP
# =============================================================================

for name, path in DATASETS.items():

    print(f"\n{'='*80}")
    print(f"PROCESSING {name}")
    print(f"{'='*80}")

    # -------------------------------------------------------------------------
    # LOAD + CLEAN
    # -------------------------------------------------------------------------

    df = pd.read_csv(path)
    print(f"Loaded {len(df):,} rows")

    # Remove calendar spreads
    df = df[~df["symbol"].str.contains("-", regex=False)].copy()

    # Parse timestamp
    df["ts_event"] = pd.to_datetime(df["ts_event"], utc=True)

    # Assign futures trading day
    df["date"] = assign_futures_trading_day(df["ts_event"])

    # Restrict sample period
    df = df[
        (df["date"] >= pd.to_datetime(START_DATE).date()) &
        (df["date"] <  pd.to_datetime(END_DATE).date())
    ]

    print(f"After cleaning: {len(df):,} rows")

    # -------------------------------------------------------------------------
    # SELECT MOST LIQUID CONTRACT PER DAY
    # -------------------------------------------------------------------------

    daily_volume = (
        df.groupby(["date", "symbol"])["volume"]
          .sum()
          .reset_index()
    )

    most_liquid = daily_volume.loc[
        daily_volume.groupby("date")["volume"].idxmax()
    ][["date", "symbol"]]

    # Keep only the front contract each day
    df_front = df.merge(most_liquid, on=["date", "symbol"], how="inner")

    # -------------------------------------------------------------------------
    # BUILD DAILY OHLCV
    # -------------------------------------------------------------------------

    daily = (
        df_front
        .sort_values("ts_event")
        .groupby("date")
        .agg(
            open   = ("open", "first"),
            high   = ("high", "max"),
            low    = ("low", "min"),
            close  = ("close", "last"),
            volume = ("volume", "sum"),
        )
        .reset_index()
    )

    daily["date"] = pd.to_datetime(daily["date"])

    # -------------------------------------------------------------------------
    # SAVE
    # -------------------------------------------------------------------------

    out_path = os.path.join(OUTPUT_DIR, f"{name}_D.csv")
    daily.to_csv(out_path, index=False)

    print(f"Saved {len(daily):,} daily bars → {out_path}")
    print(daily.head())

In [ ]:
import pandas as pd
import os

# =============================================================================
# CONFIG
# =============================================================================

INPUT_PATH  = "../data/raw/VIX.csv"
OUTPUT_DIR  = "../data/processed"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "VIX_D.csv")

START_DATE = "2019-01-01"
END_DATE   = "2026-01-01"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# LOAD
# =============================================================================

vix = pd.read_csv(INPUT_PATH)

# Parse date (MM/DD/YYYY)
vix["date"] = pd.to_datetime(vix["DATE"], format="%m/%d/%Y")

# Rename columns to match convention
vix = vix.rename(columns={
    "OPEN":  "open",
    "HIGH":  "high",
    "LOW":   "low",
    "CLOSE": "close",
})

# Keep only required columns
vix = vix[["date", "open", "high", "low", "close"]]

# Restrict period
vix = vix[
    (vix["date"] >= START_DATE) &
    (vix["date"] <  END_DATE)
]

# Sort and save
vix = vix.sort_values("date").reset_index(drop=True)
vix.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(vix):,} VIX daily bars → {OUTPUT_PATH}")
print(vix.head())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
import os

# =============================================================================
#  GLOBAL STYLE  —  light academic palette  (matches eda_professional_plots.py)
# =============================================================================
BG          = "#FFFFFF"
PANEL_BG    = "#F7F8FA"
BORDER      = "#D0D5DD"
TEXT_PRI    = "#1A1D23"
TEXT_MUT    = "#6B7280"
ACCENT_BLUE = "#2563EB"
ACCENT_RED  = "#DC2626"
ACCENT_AMB  = "#D97706"
ACCENT_GRN  = "#16A34A"
ACCENT_PUR  = "#7C3AED"

# Per-dataset colour identity
DATASET_COLOR = {
    "ES":    ACCENT_BLUE,
    "DXY":   ACCENT_GRN,
    "VIX":   ACCENT_RED,
    "MACRO": ACCENT_PUR,
}

plt.rcParams.update({
    "figure.facecolor":   BG,
    "axes.facecolor":     PANEL_BG,
    "axes.edgecolor":     BORDER,
    "axes.labelcolor":    TEXT_PRI,
    "axes.titlecolor":    TEXT_PRI,
    "axes.grid":          True,
    "grid.color":         BORDER,
    "grid.linewidth":     0.55,
    "grid.alpha":         0.7,
    "grid.linestyle":     "--",
    "xtick.color":        TEXT_MUT,
    "ytick.color":        TEXT_MUT,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.titlepad":      10,
    "axes.labelsize":     10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "font.family":        "DejaVu Sans",
    "text.color":         TEXT_PRI,
    "legend.facecolor":   BG,
    "legend.edgecolor":   BORDER,
    "legend.fontsize":    9,
    "lines.linewidth":    1.4,
    "savefig.facecolor":  BG,
    "savefig.dpi":        180,
    "figure.dpi":         110,
})

# =============================================================================
#  CONFIG
# =============================================================================
DATASETS = {
    "ES":    "../data/processed/ES_D.csv",
    "DXY":   "../data/processed/DXY_D.csv",
    "VIX":   "../data/processed/VIX_D.csv",
    "MACRO": "../data/processed/macro_D.csv",
}

VIS_DIR = "../data/visuals"
os.makedirs(VIS_DIR, exist_ok=True)

START = "2019-01-01"
END   = "2026-01-01"

# =============================================================================
#  SHARED HELPERS
# =============================================================================

# def :
#     fig.text(0.99, 0.005, "Confidential Research", fontsize=7,
#              color=TEXT_MUT, ha="right", va="bottom", alpha=0.5, style="italic")


def _style_date_axis(ax, df_dates):
    """Rotate & auto-format the x-axis for time series."""
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")


def _spine_color(ax):
    ax.spines["bottom"].set_color(BORDER)
    ax.spines["left"].set_color(BORDER)


def savefig(fig, path):
    
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


def basic_checks(df, name):
    print(f"\n{'─'*50}")
    print(f"  {name}  BASIC CHECKS")
    print(f"{'─'*50}")
    print(df.info())
    miss = df.isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    if miss.empty:
        print("  No missing values.")
    else:
        print("\nMissing value ratios:")
        print(miss.to_string())


def calendar_checks(df, name):
    dates    = pd.Series(df["date"].unique()).sort_values()
    expected = pd.bdate_range(dates.min(), dates.max())
    missing  = expected.difference(dates)
    print(f"\n  {name} calendar: {len(missing)} missing business days")
    if len(missing):
        print("  First few:", list(missing[:5]))


# =============================================================================
#  MARKET EDA  (ES / DXY)
# =============================================================================

def eda_market(name, path):
    col   = DATASET_COLOR.get(name, ACCENT_BLUE)
    out   = os.path.join(VIS_DIR, name)
    os.makedirs(out, exist_ok=True)

    df = pd.read_csv(path, parse_dates=["date"])
    df = df[(df["date"] >= START) & (df["date"] < END)].sort_values("date")

    basic_checks(df, name)
    calendar_checks(df, name)

    df = df[df["close"] > 0].copy()
    df["ret"]         = np.log(df["close"]).diff()
    df["vol_7d"]      = df["ret"].rolling(7).std()
    df["vol_14d"]     = df["ret"].rolling(14).std()
    df["vol_14d_ann"] = df["vol_14d"] * np.sqrt(252)

    # ── 1. PRICE ─────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.fill_between(df["date"], df["close"],
                    alpha=0.12, color=col, zorder=2)
    ax.plot(df["date"], df["close"], color=col, linewidth=1.5, zorder=3)
    ax.set_title(f"{name} - Close Price")
    ax.set_ylabel("Price")
    _style_date_axis(ax, df["date"])
    _spine_color(ax)
    # fig.suptitle(f"{name} Exploratory Analysis", fontsize=14,
    #              fontweight="bold", y=1.02, color=TEXT_PRI)
    savefig(fig, f"{out}/price.png")

    # ── 2. RETURNS ───────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 4))
    pos = df["ret"].clip(lower=0)
    neg = df["ret"].clip(upper=0)
    ax.fill_between(df["date"], pos, alpha=0.45, color=ACCENT_GRN,
                    label="Positive", zorder=2)
    ax.fill_between(df["date"], neg, alpha=0.45, color=ACCENT_RED,
                    label="Negative", zorder=2)
    ax.axhline(0, color=BORDER, linewidth=0.8)
    ax.legend(framealpha=0.9)
    ax.set_title(f"{name} - Log Returns")
    ax.set_ylabel("Log Return")
    _style_date_axis(ax, df["date"])
    _spine_color(ax)
    savefig(fig, f"{out}/returns.png")

    # ── 3. VOLATILITY ────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(df["date"], df["vol_7d"],  color=col,        label="7-day",  alpha=0.75)
    ax.plot(df["date"], df["vol_14d"], color=ACCENT_AMB, label="14-day", linewidth=1.5)
    ax.fill_between(df["date"], df["vol_14d"],
                    alpha=0.10, color=ACCENT_AMB, zorder=1)
    ax.legend(framealpha=0.9)
    ax.set_title(f"{name} - Rolling Volatility")
    ax.set_ylabel("Std Dev (daily returns)")
    _style_date_axis(ax, df["date"])
    _spine_color(ax)
    savefig(fig, f"{out}/volatility.png")

    # ── 4. RETURN DISTRIBUTION ───────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ret_clean = df["ret"].dropna()
    counts, edges = np.histogram(ret_clean, bins=80)
    norm_h = counts / counts.max()
    for x, cnt, nh, lo, hi in zip(
            0.5 * (edges[:-1] + edges[1:]), counts, norm_h,
            edges[:-1], edges[1:]):
        rgba = plt.matplotlib.colors.to_rgba(col, alpha=0.25 + 0.60 * nh)
        ax.bar(x, cnt, width=(hi - lo) * 0.90, color=rgba, linewidth=0, zorder=3)

    kde    = gaussian_kde(ret_clean, bw_method="scott")
    xr     = np.linspace(ret_clean.min(), ret_clean.max(), 400)
    kscale = counts.max() / kde(xr).max()
    ax.plot(xr, kde(xr) * kscale, color=col, linewidth=2.2, zorder=5)
    ax.fill_between(xr, kde(xr) * kscale, alpha=0.10, color=col, zorder=4)

    # Normal reference
    from scipy.stats import norm as sp_norm
    mu_r, sd_r = ret_clean.mean(), ret_clean.std()
    norm_pdf   = sp_norm.pdf(xr, mu_r, sd_r)
    ax.plot(xr, norm_pdf * kscale / norm_pdf.max() * (counts.max() / counts.max()),
            color=TEXT_MUT, linewidth=1.2, linestyle="--",
            label="Normal ref.", zorder=5, alpha=0.6)

    # Stats badge
    ax.text(0.97, 0.97,
            f"μ = {mu_r:.4f}\nσ = {sd_r:.4f}\n"
            f"Skew = {ret_clean.skew():.2f}\nKurt = {ret_clean.kurtosis():.2f}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=8.5, color=TEXT_MUT, fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.45", facecolor=BG,
                      edgecolor=BORDER, linewidth=0.8))
    ax.legend(framealpha=0.9)
    ax.set_title(f"{name} - Return Distribution")
    ax.set_xlabel("Log Return")
    ax.set_ylabel("Frequency")
    _spine_color(ax)
    savefig(fig, f"{out}/return_dist.png")

    # ── 5. VOLUME ────────────────────────────────────────────────────────────
    if "volume" in df.columns:
        fig, ax = plt.subplots(figsize=(11, 3.8))
        ma = df["volume"].rolling(20).mean()
        ax.bar(df["date"], df["volume"], color=col, alpha=0.35,
               width=1, zorder=2, label="Daily volume")
        ax.plot(df["date"], ma, color=col, linewidth=1.6,
                zorder=3, label="20-day MA")
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M" if x >= 1e6 else f"{int(x):,}"))
        ax.legend(framealpha=0.9)
        ax.set_title(f"{name} - Volume")
        ax.set_ylabel("Volume")
        _style_date_axis(ax, df["date"])
        _spine_color(ax)
        savefig(fig, f"{out}/volume.png")

    # ── 6. TAIL EVENTS ───────────────────────────────────────────────────────
    tail_q = df["ret"].quantile(0.01)
    is_tail = df["ret"] <= tail_q

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.fill_between(df["date"], df["ret"],
                    alpha=0.18, color=col, zorder=2)
    ax.plot(df["date"], df["ret"], color=col, linewidth=0.9,
            alpha=0.7, zorder=3)
    ax.scatter(df.loc[is_tail, "date"], df.loc[is_tail, "ret"],
               color=ACCENT_RED, s=30, zorder=5,
               label=f"1st percentile (≤ {tail_q:.4f})", edgecolors="white",
               linewidths=0.4)
    ax.axhline(tail_q, linestyle="--", color=ACCENT_RED,
               linewidth=1.1, alpha=0.7)
    ax.legend(framealpha=0.9)
    ax.set_title(f"{name} - Extreme Negative Returns (1st Percentile)")
    ax.set_ylabel("Log Return")
    _style_date_axis(ax, df["date"])
    _spine_color(ax)
    savefig(fig, f"{out}/tail_events.png")

    # ── 7. EXTREME DAY TABLE ─────────────────────────────────────────────────
    extremes = df.nsmallest(20, "ret")[["date", "ret", "vol_14d_ann", "volume"]
                                       if "volume" in df.columns
                                       else ["date", "ret", "vol_14d_ann"]]
    extremes.to_csv(f"{out}/extreme_days.csv", index=False)
    print(f"  Extreme days saved → {out}/extreme_days.csv")


# =============================================================================
#  VIX EDA
# =============================================================================

def eda_vix(path):
    name = "VIX"
    col  = DATASET_COLOR[name]
    out  = os.path.join(VIS_DIR, name)
    os.makedirs(out, exist_ok=True)

    df = pd.read_csv(path, parse_dates=["date"])
    df = df[(df["date"] >= START) & (df["date"] < END)].sort_values("date")

    basic_checks(df, name)
    calendar_checks(df, name)

    mu    = df["close"].rolling(20).mean()
    sigma = df["close"].rolling(20).std().replace(0, np.nan)
    df["vix_z"] = (df["close"] - mu) / sigma

    # ── 1. VIX LEVEL ─────────────────────────────────────────────────────────
    # Regime bands: calm <15, elevated 15-25, stress >25
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.axhspan(0,  15, alpha=0.07, color=ACCENT_GRN, zorder=1)
    ax.axhspan(15, 25, alpha=0.07, color=ACCENT_AMB, zorder=1)
    ax.axhspan(25, df["close"].max() * 1.1,
               alpha=0.07, color=ACCENT_RED, zorder=1)
 
    for level, label, clr in [(15, "Calm / Elevated", ACCENT_GRN),
                               (25, "Elevated / Stress", ACCENT_AMB)]:
        ax.axhline(level, linestyle="--", linewidth=0.9,
                   color=clr, alpha=0.6)
        ax.text(df["date"].iloc[-1], level + 0.3, f" {level}",
                fontsize=8, color=clr, va="bottom")
 
    ax.fill_between(df["date"], df["close"],
                    alpha=0.15, color=col, zorder=2)
    ax.plot(df["date"], df["close"], color=col, linewidth=1.5, zorder=3)
 
    legend_handles = [
        mpatches.Patch(color=col,        label="VIX Close"),
        mpatches.Patch(facecolor=ACCENT_GRN, alpha=0.35, label="Calm  (< 15)"),
        mpatches.Patch(facecolor=ACCENT_AMB, alpha=0.35, label="Elevated  (15 – 25)"),
        mpatches.Patch(facecolor=ACCENT_RED, alpha=0.35, label="Stress  (> 25)"),
    ]
    ax.legend(handles=legend_handles, loc="upper right",
              framealpha=0.9, fontsize=8.5)
 
    ax.set_title("VIX - Closing Level with Regime Bands")
    ax.set_ylabel("VIX")
    _style_date_axis(ax, df["date"])
    _spine_color(ax)
    # fig.suptitle("VIX Exploratory Analysis", fontsize=14,
    #              fontweight="bold", y=1.02, color=TEXT_PRI)
    savefig(fig, f"{out}/level.png")

    # ── 2. Z-SCORE ───────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 4))
    pos = df["vix_z"].clip(lower=0)
    neg = df["vix_z"].clip(upper=0)
    ax.fill_between(df["date"], pos, alpha=0.45, color=ACCENT_RED,
                    label="Above avg", zorder=2)
    ax.fill_between(df["date"], neg, alpha=0.35, color=ACCENT_BLUE,
                    label="Below avg", zorder=2)
    ax.axhline(0,  color=BORDER,     linewidth=0.8)
    ax.axhline(2,  color=ACCENT_RED, linewidth=1.0, linestyle="--", alpha=0.7)
    ax.axhline(-2, color=ACCENT_RED, linewidth=1.0, linestyle="--", alpha=0.7)
    ax.text(df["date"].iloc[-1], 2.1,  " +2σ", fontsize=8, color=ACCENT_RED)
    ax.text(df["date"].iloc[-1], -2.35, " −2σ", fontsize=8, color=ACCENT_RED)
    ax.legend(framealpha=0.9)
    ax.set_title("VIX - 20-Day Rolling Z-Score")
    ax.set_ylabel("Z-Score")
    _style_date_axis(ax, df["date"])
    _spine_color(ax)
    savefig(fig, f"{out}/zscore.png")

    # ── 3. DISTRIBUTION ──────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4.5))
    data   = df["close"].dropna()
    counts, edges = np.histogram(data, bins=60)
    norm_h = counts / counts.max()
    for x, cnt, nh, lo, hi in zip(
            0.5 * (edges[:-1] + edges[1:]), counts, norm_h,
            edges[:-1], edges[1:]):
        rgba = plt.matplotlib.colors.to_rgba(col, alpha=0.25 + 0.60 * nh)
        ax.bar(x, cnt, width=(hi - lo) * 0.90, color=rgba, linewidth=0, zorder=3)

    kde    = gaussian_kde(data, bw_method="scott")
    xr     = np.linspace(data.min(), data.max(), 400)
    kscale = counts.max() / kde(xr).max()
    ax.plot(xr, kde(xr) * kscale, color=col, linewidth=2.2, zorder=5)
    ax.fill_between(xr, kde(xr) * kscale, alpha=0.10, color=col, zorder=4)

    for pct, label in [(data.quantile(0.75), "Q3"),
                       (data.quantile(0.90), "P90"),
                       (data.quantile(0.95), "P95")]:
        ax.axvline(pct, linestyle="--", linewidth=1.0,
                   color=ACCENT_AMB, alpha=0.75)
        ax.text(pct, counts.max() * 0.97, f" {label}\n {pct:.1f}",
                fontsize=7.5, color=ACCENT_AMB, va="top", rotation=90)

    ax.text(0.97, 0.97,
            f"μ = {data.mean():.2f}\nσ = {data.std():.2f}\n"
            f"Skew = {data.skew():.2f}\nKurt = {data.kurtosis():.2f}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=8.5, color=TEXT_MUT, fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.45", facecolor=BG,
                      edgecolor=BORDER, linewidth=0.8))
    ax.set_title("VIX - Level Distribution")
    ax.set_xlabel("VIX")
    ax.set_ylabel("Frequency")
    _spine_color(ax)
    savefig(fig, f"{out}/distribution.png")


# =============================================================================
#  MACRO EDA
# =============================================================================

def eda_macro(path):
    name = "MACRO"
    col  = DATASET_COLOR[name]
    out  = os.path.join(VIS_DIR, name)
    os.makedirs(out, exist_ok=True)

    df = pd.read_csv(path, parse_dates=["date"])
    df = df[(df["date"] >= START) & (df["date"] < END)].sort_values("date")

    basic_checks(df, name)

    macro_cols = [c for c in df.columns if c != "date"]

    # ── 1. INDIVIDUAL SERIES ─────────────────────────────────────────────────
    for series_col in macro_cols:
        series = df[["date", series_col]].dropna()
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.fill_between(series["date"], series[series_col],
                        alpha=0.12, color=col, zorder=2)
        ax.plot(series["date"], series[series_col],
                color=col, linewidth=1.5, zorder=3)
        ax.set_title(f"Macro - {series_col}")
        ax.set_ylabel(series_col)
        _style_date_axis(ax, series["date"])
        _spine_color(ax)
        savefig(fig, f"{out}/{series_col}.png")

    # ── 2. MULTI-PANEL OVERVIEW ───────────────────────────────────────────────
    n      = len(macro_cols)
    ncols  = 2
    nrows  = int(np.ceil(n / ncols))
    colors = [ACCENT_BLUE, ACCENT_RED, ACCENT_AMB, ACCENT_GRN,
              ACCENT_PUR, "#0891B2", "#BE185D", "#065F46"]

    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.0 * nrows))
    fig.subplots_adjust(top=0.90, bottom=0.07, left=0.07,
                        right=0.97, hspace=0.60, wspace=0.28)
    axes_flat = axes.flatten() if nrows > 1 else list(axes)

    for idx, series_col in enumerate(macro_cols):
        ax     = axes_flat[idx]
        series = df[["date", series_col]].dropna()
        c      = colors[idx % len(colors)]
        ax.fill_between(series["date"], series[series_col],
                        alpha=0.15, color=c, zorder=2)
        ax.plot(series["date"], series[series_col],
                color=c, linewidth=1.4, zorder=3)
        ax.set_title(series_col)
        ax.set_ylabel(series_col, fontsize=8)
        _style_date_axis(ax, series["date"])
        _spine_color(ax)
        ax.tick_params(axis="x", labelsize=7)
        ax.tick_params(axis="y", labelsize=8)

    for ax in axes_flat[n:]:
        ax.set_visible(False)

    fig.suptitle("Macro Features - Overview", fontsize=15,
                 fontweight="bold", color=TEXT_PRI, y=0.97)
    
    fig.savefig(f"{out}/all_series_overview.png", bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out}/all_series_overview.png")

    # ── 3. CORRELATION HEATMAP ────────────────────────────────────────────────
    corr = (df[macro_cols]
            .replace([np.inf, -np.inf], np.nan)
            .corr()
            .fillna(0))
    m = len(macro_cols)

    cmap = LinearSegmentedColormap.from_list(
        "academic_div",
        ["#2563EB", "#93C5FD", "#F3F4F6", "#FCA5A5", "#DC2626"],
        N=256)

    fig, ax = plt.subplots(figsize=(max(8, m * 0.95), max(6, m * 0.85)))
    fig.subplots_adjust(top=0.90, bottom=0.16, left=0.16, right=0.91)
    im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")

    for i in range(m):
        for j in range(m):
            val = corr.values[i, j]
            text_color = TEXT_PRI if abs(val) < 0.55 else "white"
            weight     = "bold" if i != j and abs(val) > 0.5 else "normal"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=8, color=text_color, fontweight=weight)

    for k in np.arange(-0.5, m, 1):
        ax.axhline(k, color=BG, linewidth=1.6, zorder=5)
        ax.axvline(k, color=BG, linewidth=1.6, zorder=5)

    for d in range(m):
        ax.add_patch(plt.Rectangle(
            (d - 0.5, d - 0.5), 1, 1,
            fill=False, edgecolor=ACCENT_AMB, linewidth=1.1, zorder=6))

    ax.set_xticks(range(m))
    ax.set_yticks(range(m))
    ax.set_xticklabels(macro_cols, rotation=45, ha="right",
                       fontsize=9, color=TEXT_PRI)
    ax.set_yticklabels(macro_cols, fontsize=9, color=TEXT_PRI)
    ax.tick_params(top=False, bottom=False, left=False, right=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.ax.yaxis.set_tick_params(color=TEXT_MUT, labelsize=9)
    cbar.outline.set_edgecolor(BORDER)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_MUT)
    cbar.set_label("Pearson r", color=TEXT_MUT, fontsize=10, labelpad=8)

    fig.suptitle("Macro Features - Correlation Matrix", fontsize=14,
                 fontweight="bold", color=TEXT_PRI, y=0.97)
    
    fig.savefig(f"{out}/correlation.png", bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out}/correlation.png")


# =============================================================================
#  RUN ALL
# =============================================================================
if __name__ == "__main__":
    eda_market("ES",  DATASETS["ES"])
    eda_market("DXY", DATASETS["DXY"])
    eda_vix(DATASETS["VIX"])
    eda_macro(DATASETS["MACRO"])